## Binding Library (Walkthrough)

Original code by Dan Redeker, adapted by Sarah Hong

### Step 1: Initialize Sequence Space

Given a desired sequence length of $x$ base pairs, create all possible sequences using Watson-Crick DNA base pairing rules. Split those into sequence and complementary spaces, say $S$ and $S'$. Note: Here, please provide your own temperature space with "Sequence", "Complement", and "Melting Temp" columns for your base pairs in question.

Then, we filter the space based on the following criteria:

1. #GC/#total base pairs in sequence must be within some range
2. No palindromes (eg, "AATGCATT" is not allowed because it could fold in on itself and self-bind.)
3. Melting temperature of the sequence must be within some allowed range
4. Sequences with "GGG" or "CCC" should be excluded

In [1]:
from itertools import product
import pandas as pd

class SequenceSpace:
    def __init__(self, seq_length: int=8):
        """Initialize our binding library"""
        self.BASES = "ATGC"
        self.COMPLEMENT = str.maketrans("ATGC", "TACG")
        self.SEQ_LENGTH = seq_length
        self.init_temp_space()
        self.init_space(seq_length)
        self.N = len(self.s1)

    def init_temp_space(self):
        """Initialize the melting temperature map from the excel file
        with all melting temps for sequences of length SEQ_LENGTH
        
        Final output:
            -> self.tm[sequence] = melting temp
        """
        # read in the excel file with melting temps
        self.tm_df = pd.read_excel(f"data/melting_temps/{self.SEQ_LENGTH}bp.xlsx")
        self.tm = {}
        for _, row in self.tm_df.iterrows():
            seq = row['Sequence']
            comp = row['Complement']
            temp = row['Melting Temp']
            self.tm[seq] = temp
            self.tm[comp] = temp

    def init_space(self, seq_length: int):
        """Initialize the binding space. Creates s1 and s2, which are lists of the
        color sequences and their complements respectively."""
        self.seq_length = seq_length
        sequences = [''.join(p) for p in product(self.BASES, repeat=seq_length)]
        self.s1, self.s2 = self.sort_sequences(sequences)

    def complement(self, seq: str, reverse=True) -> str:
        """Return the complement of a sequence"""
        # we reverse cus the excel reverses it
        comp = seq.translate(self.COMPLEMENT)
        return comp if not reverse else comp[::-1]

    def sort_sequences(self, sequences: list[str]) -> tuple[list[str], list[str]]:
        """Sort sequences into color and complements"""
        seen = set()
        s1, s2 = [], []
        for s in sequences:
            if s in seen:
                continue
            c = self.complement(s, reverse=True)
            # choose a canonical representative
            color, comp = (s, c) if s < c else (c, s)

            # add to our lists and seen set
            s1.append(color)
            s2.append(comp)
            seen.add(s)
            seen.add(c)

        return s1, s2
    
    def filter_sequences(self, gc_range: tuple[float, float]=(0.3, 0.7), 
                         temp_range: tuple[float, float]=(5.0, 50.0), 
                         exclude_ggg=True):
        """Filter the sequence space based on the provided criteria.
        
        Args:
            gc_range: Min/max GC proportion per sequence
            temp_range: Min/max melting temperature per sequence
            exclude_ggg: Whether to exclude sequences containing "GGG" or "CCC"
        """
        s1, s2 = [], []
        for s, c in zip(self.s1, self.s2):
            
            good_ggg = self._excludes_ggg(s, exclude_ggg)
            good_gc = self._filter_gc(s, gc_range)
            good_temp = self._filter_temp(s, temp_range)
            good_palindrome = not self._is_palindrome(s)

            if good_gc and good_temp and good_ggg and good_palindrome:
                s1.append(s)
                s2.append(c)
        
        print(f"Filtered from {self.N} to {len(s1)} sequences.")
        self.N = len(s1)
        self.s1, self.s2 = s1, s2

    def _filter_gc(self, s: str, gc_range: tuple[float, float]=(0.3, 0.7)):
        """Returns True if the sequence 's' passes the gc_range filter check."""
        gc_count = s.count("G") + s.count("C")
        gc_frac = gc_count / self.SEQ_LENGTH

        if gc_frac >= gc_range[0] and gc_frac <= gc_range[1]:
            return True
        return False

    def _excludes_ggg(self, s: str, exclude_ggg=True):
        """Returns True if the sequence 's' passes the excludes "GGG" check."""
        if not exclude_ggg:
            return True
        
        if "GGG" in s or "CCC" in s:
            return False 
        return True

    def _filter_temp(self, s: str, temp_range: tuple[float, float]=(5, 50)):
        """Returns True if the sequence 's' passes the melting temperature range check."""
        temp = self.tm.get(s, None)
        if not temp:
            return False
        if temp_range[0] <= temp <= temp_range[1]:
            return True
        return False
    
    def _is_palindrome(self, s: str) -> bool:
        """Check if a sequence is a palindrome"""
        return s == self.complement(s)

### Step 2: Initialize Graph

Define two DNA sequences $s_1, s_2$ are **orthogonal** if they have no contiguous complementary overlap of over $k$ base pairs, for some parameter $k$. 
- For speed, the code assumes any sequence pair with a shared $k+1$-length subsequence is not orthogonal. (Further extensions could consider the number of shared subsequences; we don't.)

Using this definition, we create a *graph* of our sequence space, where:
- Nodes = sequences
- Edge = present if those two nodes are *NOT* orthogonal with each other

Then create two adjacency matrices: AA and AB such that
- AA - Orthogonality between all sequences in $S_1$
- AB - Orthogonality between all sequences in $S_2$

Suffices to take $C = (AA + AB)$ as our final adjacency matrix to account for cross-complementary interactions across all binding scenarios between two sequences.

### Step 3: Greedy Maximum Independent Set

Then we run a greedy algorithm to find the maximum independent set of nodes from our graph $C$, which returns a set of nodes which are all guaranteed to not be orthogonal with each other. Algorithm details are in the code, and was adapted from Ballard-Myer's notes here: https://www.gcsu.edu/sites/files/page-assets/node-808/attachments/ballardmyer.pdf


In [2]:
from collections import defaultdict

class OrthoGraph:
    def __init__(self, ss: SequenceSpace, k: int):
        self.ss = ss
        self.k = k
        self.t = k+1 # forbidden k-mer length
        self.init_neighbors()

        print(f"Initialized graph for {self.ss.N} sequences with k={self.k}.")

    def _tmers(self, s:str) -> set[str]:
        """Return the set of all t-mers in the sequence s.
        Rk: t-mer is a substring of length t, where t = k+1 (the forbidden k-mer length)"""
        return {s[i:i+self.t] for i in range(len(s) - self.t + 1)}

    def init_neighbors(self):
        """
        Compute the neighbors for each sequence. Suffices to compute all
        t=k+1-mers for each sequence, then declare as neighbors any sequences 
        that share any t-mer.

        Fills in self.neighbors and self.self_conflicting in place.
        """
        s1 = self.ss.s1
        s2 = self.ss.s2
        N = self.ss.N

        self.neighbors = [set() for _ in range(N)]
        self.self_conflicting = set() # track sequences that could bind to themselves

        # precompute t-mers for every sequence
        t_mers_s1 = [self._tmers(s) for s in s1]
        t_mers_s2 = [self._tmers(s) for s in s2]

        # create a map from t-mer to the set of sequences which contain it
        tmer_to_s1 = defaultdict(set) # {t_mer: set of indices in s1 that contain t_mer}
        tmer_to_s2 = defaultdict(set) # ... for s2
        for s in range(N):
            for t_mer in t_mers_s1[s]:
                tmer_to_s1[t_mer].add(s)
            for t_mer in t_mers_s2[s]:
                tmer_to_s2[t_mer].add(s)

        # find neighbors based on shared t-mers
        # achieves the logical-OR of conflicts from (s1 x s1) and (s1 x s2)
        for s in range(N): # for string s in s1
            bad = set()
            for t_mer in t_mers_s1[s]:
                bad |= tmer_to_s1[t_mer] # AA conflicts
                bad |= tmer_to_s2[t_mer] # AB conflicts

            # > detect self-complementarity
            if t_mers_s1[s] & t_mers_s2[s]:
                self.self_conflicting.add(s)
                bad.add(s)

            self.neighbors[s] = bad


    def is_orthogonal(self, s1: str, s2: str):
        return s1 not in self.neighbors[s2]
    
    def get_orthogonal_sequences(self) -> tuple[list[str], list[str]]:
        """
        Greedy algorithm to find a large set of orthogonal sequences.
        
        Starts from the sequence with the fewest neighbors, adds it to the orthogonal
        set, removes its neighboring nodes and their incident edges, and repeats until no 
        sequences remain.
        """
        remaining = set(range(self.ss.N)) - self.self_conflicting
        orth1, orth2 = [], []

        while remaining:
            best = min(remaining, key=lambda s: len(self.neighbors[s] & remaining))
            orth1.append(self.ss.s1[best])
            orth2.append(self.ss.s2[best])

            # remove chosen vertex and all its conflicts
            remaining -= self.neighbors[best]
            remaining.discard(best)

        return orth1, orth2
    
    def export_orthogonal_set(self, filename: str):
        """Export the orthogonal set to a CSV file with columns "Sequence" and "Complement"."""
        orth1, orth2 = self.get_orthogonal_sequences()
        df = pd.DataFrame({"Sequence": orth1, "Complement": orth2})
        df.to_csv(filename, index=False)
        print(f"Exported {len(orth1)} orthogonal sequences to {filename}.")

### Usage

To use the functions, initialize a `SequenceSpace`, filter based on your desired criteria, create an `OrthoGraph` object with that sequence space and some desired $k$ for max base pair overlap, then call `get_orthogonal_sequences()` to get your library! 

Note: Can also export to CSV with `export_orthogonal_set(filename)`.

In [ ]:
# test the sequence space initialization
ss = SequenceSpace(8)
ss.filter_sequences(gc_range=(0.3, 0.7), temp_range=(5.0, 50.0), exclude_ggg=True)
print(f"Seq1 contains {len(ss.s1)} strands. Seq2 contains {len(ss.s2)} strands.")

ortho_graph = OrthoGraph(ss, k=3)
orthogonal_seqs = ortho_graph.get_orthogonal_sequences()
print(f"Found {len(orthogonal_seqs[0])} orthogonal sequences.")

ortho_graph.export_orthogonal_set("results/8bp_ortho_lib.csv")

Filtered from 32896 to 20576 sequences.
Seq1 contains 20576 strands. Seq2 contains 20576 strands.
Initialized graph for 20576 sequences with k=3.
Found 26 orthogonal sequences.
Exported 26 orthogonal sequences to 8bp_ortho_lib.csv.


In [4]:
# a small sanity check within here!
for s1, s2 in zip(orthogonal_seqs[0], orthogonal_seqs[1]):
    print(f"seq: {s1}, comp: {s2}")

seq: AGAGAGAG, comp: CTCTCTCT
seq: ACACACAC, comp: GTGTGTGT
seq: AGGAGGAG, comp: CTCCTCCT
seq: AGTAGTAG, comp: CTACTACT
seq: ACCACCAC, comp: GTGGTGGT
seq: AGTGAGTG, comp: CACTCACT
seq: CATCATCA, comp: TGATGATG
seq: ATGTATGG, comp: CCATACAT
seq: GATAATGC, comp: GCATTATC
seq: AATAGGCG, comp: CGCCTATT
seq: CGTGCCAA, comp: TTGGCACG
seq: CGGCTAAA, comp: TTTAGCCG
seq: AGCAGCAG, comp: CTGCTGCT
seq: AAGCGAGC, comp: GCTCGCTT
seq: CTTACGCA, comp: TGCGTAAG
seq: GCAATCTA, comp: TAGATTGC
seq: ACCTGGAT, comp: ATCCAGGT
seq: CAAAAAGG, comp: CCTTTTTG
seq: ATTTCCGC, comp: GCGGAAAT
seq: AAACCGAT, comp: ATCGGTTT
seq: ACGACGAC, comp: GTCGTCGT
seq: AGAAGAAG, comp: CTTCTTCT
seq: CCGTTCAG, comp: CTGAACGG
seq: ACTTGACC, comp: GGTCAAGT
seq: ACAACAAC, comp: GTTGTTGT
seq: AACTGTCT, comp: AGACAGTT
